# Week 5, Day 1: Google ADK, A2A, and MCP

## What this lab covers

This lab configures a Google ADK agent through LiteLLM and Azure OpenAI, then runs it with an in-memory runner. The exercises add a shared SQLite task board, turn board operations and Outlook email into agent tools, connect a filesystem toolset through MCP, and use the tools to complete a goal in an agent loop.

### Day 01: Google ADK + A2A

In [1]:
## importing required libraries
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.models.lite_llm import LiteLlm
# from quiet import silence
# silence() # turn down some noisy library logging so the agent's trace is easy to read
load_dotenv(override=True)
import board # importing the custom board module for managing todos


In [2]:
# loading deployment name from .env file
AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
if AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI:
    print(f"Deployment available: {AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI}")
else:
    print("Deployment isn't available in .env")

Deployment available: gpt-5.4-mini


### Step 01: Create the Agent

In [3]:
# function to return LiteLlm format model deployment
def LiteLlm_model(deployment):
    return LiteLlm(model = f"azure/{deployment}")

In [4]:
# setting up an agent call using LiteLlm
agent = LlmAgent(
    model = LiteLlm(model = f"azure/{AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI}"),
    name = 'assistant',
    instruction="You're a concise, friendly assistant, Reply in a single short sentence."
)


### Step 02: Run prepared agent instance

In [7]:
# using InMemoryRunner in Google ADK
result = await InMemoryRunner(
    agent = agent
).run_debug("Say hello in French.", verbose=True)

01:15:36 - LiteLLM:WARNING: get_model_cost_map.py:264 - LiteLLM: model cost map fetch attempt 1/3 failed (ConnectError fetching https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [WinError 10054] An existing connection was forcibly closed by the remote host); retrying in 2.2s
01:15:42 - LiteLLM:WARNING: get_model_cost_map.py:264 - LiteLLM: model cost map fetch attempt 2/3 failed (ConnectError fetching https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [WinError 10054] An existing connection was forcibly closed by the remote host); retrying in 4.7s
01:15:50 - LiteLLM:WARNING: get_model_cost_map.py:509 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: ConnectError fetching https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [WinError 10054] An existing connection was fo

assistant > Bonjour !


In [9]:
# using InMemoryRunner in Google ADK
result = await InMemoryRunner(
    agent = agent
).run_debug("Where is strait of hormus..?", verbose=True)

assistant > The Strait of Hormuz is between Iran and Oman, connecting the Persian Gulf to the Gulf of Oman.


### One Small Project this week: A SQLite todo board

In [26]:
# Reset the board and set up the initial goal for the agent
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt',
  'status': 'pending',
  'result': ''}]

In [27]:
# let's see the board
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt

### Step 03: Add Tools to the agent 

In [33]:
# let's define some functions for interacting with the board
def show_todos() -> list[dict]:
    """List every todo on board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "steps_id": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a steps or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "result": result, "status": "done"}


In [9]:
board_agent = LlmAgent(
    model=LiteLlm_model(AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI),
    name="board_agent",
    instruction="You help manage a shared todo board.",
    tools=[show_todos, complete_task]
)
result = await InMemoryRunner(
    agent=board_agent
).run_debug("What is on the board right now, and what is its status?",verbose=True)

board_agent > [Calling tool: show_todos({})]
board_agent > [Tool result: {'result': [{'id': 1, 'parent_id': None, 'task': 'Read notes.txt, translate its contents into natura...]
board_agent > On the board right now there is 1 goal:

- **Task 1:** Read `notes.txt`, translate its contents into natural Spanish, and write the Spanish to `spanish.txt`
  - **Status:** pending

No subtasks are listed.


In [5]:
## Let's try setting up a email sending tool and ask the agent to send an email.

## Setting up email sending method/tool 
# define a method/function to send email via Outlook COM
import urllib3
import requests
import re
import asyncio
import datetime
import win32com.client
import pythoncom
import subprocess
import time

def send_email_tool( 
    subject: str,
    body: str
    ) -> bool:
    """Send an email using the local Outlook desktop client."""
    # Normalize recipient to a semicolon-separated string (Outlook format)
    recipient = os.getenv("EMAIL_ADDRESS_TO")
    recipients = []
    if isinstance(recipient, list):
        recipients = [r.strip() for r in recipient if r.strip()]
    elif isinstance(recipient, str):
        if "," in recipient:
            recipients = [r.strip() for r in recipient.split(",") if r.strip()]
        else:
            recipients = [recipient.strip()]
    
    def validate_email(email: str) -> bool:
        """Validate email format."""
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        return re.match(pattern, email.strip()) is not None

    # Validate all recipients
    invalid_recipients = [r for r in recipients if not validate_email(r)]
    if invalid_recipients:
        print(f"Error: Invalid email format(s): {invalid_recipients}")
        return False
    
    recipient_str = "; ".join(recipients)
    
    if not recipient_str:
        print("Error: No recipients specified. Email not sent.")
        return False

    if body is None:
        body = "Hello there!! "

    print(f"\nSending email to {recipient_str} with subject '{subject}'...")

    # Ensure COM is initialized for the current thread (agent tool calls may run in worker threads).
    pythoncom.CoInitialize()
    try:
        # Connect to Outlook, launch it if not running
        outlook = None
        for attempt in range(5):
            try:
                outlook = win32com.client.Dispatch("Outlook.Application")
                outlook.GetNamespace("MAPI")
                print("Outlook is up & running...")
                break
            except Exception as e:
                if attempt == 0:
                    print("Outlook not running — launching...")
                    for p in [r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                              r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE"]:
                        if os.path.exists(p):
                            subprocess.Popen([p])
                            time.sleep(45)
                            break
                    else:
                        print("Error: Outlook executable not found.")
                        return False
                elif attempt < 4:
                    print(f"Outlook connect attempt {attempt + 1}/5 — waiting...")
                    time.sleep(15)
                else:
                    print(f"Error: Outlook connect failed after 5 attempts: {e}")
                    return False

        try:
            mail = outlook.CreateItem(0)
            mail.To = recipient_str
            mail.Subject = subject
            mail.Body = body

            mail.Send()
            print(f"\nEmail sent successfully to '{recipient_str}' with subject '{subject}'.")
            return True

        except Exception as e:
            print(f"Failed to send email: {e}")
            # Print recipient info for debugging
            print(f"Debug - Recipients used: {repr(recipient_str)}")
            print(f"Debug - Recipient type: {type(recipient_str)}")
            return False
    finally:
        try:
            pythoncom.CoUninitialize()
        except Exception:
            pass


notifier = LlmAgent(
    model=LiteLlm_model(AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI),
    name="notifier",
    instruction="You help send emails using Outlook.",
    tools=[send_email_tool]
)
result = await InMemoryRunner(
    agent=notifier
).run_debug("Send an email from Mr. Jagga saying 'Hello from Google ADK!!'",verbose=True)

c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\google\adk\models\llm_request.py:298: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()
00:50:29 - LiteLLM:WARNING: get_model_cost_map.py:264 - LiteLLM: model cost map fetch attempt 1/3 failed (ConnectError fetching https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [WinError 10054] An existing connection was forcibly closed by the remote host); retrying in 2.9s
00:50:37 - LiteLLM:WARNING: get_model_cost_map.py:264 - LiteLLM: model cost map fetch attempt 2/3 failed (ConnectError fetching https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [WinError 10054] An existing connection was forcibly closed by the remote host); retrying in 4.4s
00:50:47 - LiteLLM:WARNING: get_model_cost_map.py:509 - LiteLLM: Failed

notifier > [Calling tool: send_email_tool({'subject': 'Hello from Google ADK!!', 'body': 'He...)]

Sending email to hardeep.singh3@lseg.com with subject 'Hello from Google ADK!!'...
Outlook is up & running...

Email sent successfully to 'hardeep.singh3@lseg.com' with subject 'Hello from Google ADK!!'.
notifier > [Tool result: {'result': True}]
notifier > Done.


### Step 04: Add MCP (Model Context Protocol)

In [ ]:
# Initialize the MCP toolset with SSE connection parameters
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import SseConnectionParams

filesystem = McpToolset(
    connection_params=SseConnectionParams(
        url="http://127.0.0.1:8000/sse"
    )
)

In [24]:
# let's define a agent
file_agent = LlmAgent(
    model=LiteLlm_model(AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI),
    name="file_agent",
    instruction="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem]
)

In [ ]:
result = await InMemoryRunner(
    agent=file_agent
    ).run_debug("Read notes.txt and summarize it in one short sentence.", verbose=True)
    

Failed to configure mTLS using AsyncAuthorizedSession: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.


file_agent > [Calling tool: read_file({'path': 'notes.txt'})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Welcome to the team.\n\nToday we are building a small languag...]
file_agent > The notes welcome the team and explain that they are building a small language tutor one task at a time.


Error in sse_reader
Traceback (most recent call last):
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpx\_transports\default.py", line 271, in __aiter__
    async for part in self._httpcore_stream:
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpcore\_async\connection_pool.py", line 407, in __aiter__
    raise exc from None
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpcore\_async\connection_pool.py", line 403, in __aiter__
    async for part in self._stream:
  File "c:\Users\hsingh8\OneDrive - London Stock

### Step 05: Put agent in a loop with a goal 

In [34]:
# let's give this agent some serious filesystem capabilities and ask to do some file operations
from google.genai import types

INSTRUCTIONS="""
You're a careful worker with a todo board and a set of file tools.
Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into,
added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal.
Your files lives in the single folder, your tools are allowed to use.
"""

worker = LlmAgent(
    model=LiteLlm_model(AZURE_OPENAI_DEPLOYMENT_GPT_54_MINI),
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem, send_email_tool]
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.")
board.claim_todo(goal_id)

# let's drive the loop ourselves with run_sync instead of run_debug, so we can watch it:
# print each tool call, and re-draw the board whenever a step is planned or ticket off.

runner = InMemoryRunner(
    agent=worker
)
session = await runner.session_service.create_session(app_name=runner.app_name,user_id="Hardeep")

async for event in runner.run_async(
    user_id="Hardeep",
    session_id=session.id,
    new_message=types.UserContent("Please work the pending goal on the board.")
    ):
    for call in event.get_function_calls():
        print(f"tool: {call.name}({call.args})")
    if any(r.name in ("plan_steps","complete_task") for r in event.get_function_responses()):
        board.show_board()
    if event.is_final_response():
        print(event.content.parts[0].text)


tool: show_todos({})
tool: plan_steps({'goal_id': 1, 'steps': ['Read notes.txt and extract the key content', 'Translate the notes into natural German', 'Write the German translation to german.txt', 'Send an email summarizing the completed work']})


Goal #1: Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.
  Step #2: Read notes.txt and extract the key content
  Step #3: Translate the notes into natural German
  Step #4: Write the German translation to german.txt
  Step #5: Send an email summarizing the completed work

tool: read_file({'Path': 'notes.txt'})
tool: write_file({'Path': 'german.txt', 'Content': ''})
tool: read_file({'path': 'notes.txt'})
tool: write_file({'path': 'german.txt', 'content': 'Willkommen im Team.\n\nHeute bauen wir einen kleinen Sprachlehrer, Stück für Stück. Jeder Helfer\nwählt eine einzelne Aufgabe vom gemeinsamen Board aus, erledigt die Arbeit sorgfältig\nund markiert die Aufgabe als erledigt. Lies deine Aufgabe aufmerksam, nimm dir Zeit\nund gib dein Bestes.\n'})
tool: complete_task({'task_id': 2, 'result': 'Read notes.txt and extracted the source text for translation.'})


Goal #1: Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.
  Step #2: Read notes.txt and extract the key content  Read notes.txt and extracted the source text for translation.
  Step #3: Translate the notes into natural German
  Step #4: Write the German translation to german.txt
  Step #5: Send an email summarizing the completed work

tool: complete_task({'task_id': 3, 'result': 'Translated the notes into natural German.'})


...

tool: send_email_tool({'subject': 'Completed task: notes translated to German', 'body': 'Done: I read notes.txt, translated it into natural German, and wrote the result to german.txt.\n\nSummary of the translation:\n- Welcoming message to the team\n- Description of building a small language tutor step by step\n- Reminder to read tasks carefully, take time, and do your best'})

Sending email to hardeep.singh3@lseg.com with subject 'Completed task: notes translated to German'...
Outlook is up & running...

Email sent successfully to 'hardeep.singh3@lseg.com' with subject 'Completed task: notes translated to German'.
tool: complete_task({'task_id': 5, 'result': 'Sent an email summarizing the completed translation work.'})


Goal #1: Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.
  Step #2: Read notes.txt and extract the key content  Read notes.txt and extracted the source text for translation.
  Step #3: Translate the notes into natural German  Translated the notes into natural German.
  Step #4: Write the German translation to german.txt  Wrote the German translation to german.txt.
  Step #5: Send an email summarizing the completed work  Sent an email summarizing the completed translation work.

tool: complete_task({'task_id': 1, 'result': 'Translated notes.txt into German, wrote german.txt, and sent the summary email.'})


Goal #1: Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.  Translated notes.txt into German, wrote german.txt, and sent the summary email.
  Step #2: Read notes.txt and extract the key content  Read notes.txt and extracted the source text for translation.
  Step #3: Translate the notes into natural German  Translated the notes into natural German.
  Step #4: Write the German translation to german.txt  Wrote the German translation to german.txt.
  Step #5: Send an email summarizing the completed work  Sent an email summarizing the completed translation work.

Done. I translated `notes.txt` into German, saved it as `german.txt`, sent the summary email, and closed the goal.
